In [ ]:
from glob import glob
from cmcmultimodal.io import mat2nii
from pathlib import Path
import os

my_files = glob('../tests/testdata/Slice*.mat')
highres_files = []
lowres_files = []
for f in my_files:
    nii_highres, nii_lowres = mat2nii(f, nii_lowres_file=os.path.join(Path(f).parent,'lowres/',
                                      Path(f).name.replace('.mat','.nii.gz')), downsample=10)
    highres_files.append(nii_highres)
    lowres_files.append(nii_lowres)
    
highres_files

In [ ]:
# Process all MOE slides

from cmcmultimodal.proc import psoct
from pathlib import Path

datadir = '/Users/Vasilis/CMC_data/Moe'
output_path = '/Users/Vasilis/Downloads/CMC_results/Moe_cc_Ret'
mri_ref = '/Users/Vasilis/CMC_data/Moe/MRI/reoriented_FA.nii.gz'
seq_params = '/Users/Vasilis/CMC_data/Moe/PSOCT/seq_params.json'

my_data = psoct(Path(datadir), seq_params, reg_modality='Retardance', verbose=True)
# my_data = psoct(Path(datadir), seq_params, slide_range=(98,160), reg_modality='Retardance', verbose=True)
my_data.run_registration(bad_slides=[140,])

indiv_slides = my_data.run_slide_deck_creation(other_images=['Retardance', 'Cross', 'Orientation'], 
                                               output_path=output_path,
                                               mri_ref=mri_ref,
                                               downsample=1)

In [ ]:
from fsl.data.image         import Image
import numpy as np
from pathlib import Path

ref_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_5')
est_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_6')

def compare_images(ref, est):
    ref_img = Image(ref)
    est_img = Image(est)

    if not np.allclose(ref_img.data, est_img.data):
        print('\t', 'Images are NOT equal!')
    if ref_img.header != est_img.header:
        print('\t', 'Headers are NOT equal!')

def compare_matrices(ref, est):
    ref_mat = np.loadtxt(ref)
    est_mat = np.loadtxt(est)

    if not np.allclose(ref_mat, est_mat, atol=0.001):
        print('\t', 'Matrices are NOT equal!')

for file in ref_path.glob('*'):
    if file.is_dir():
        for subfile in file.glob('*'):
            print(subfile)
            corresponding_est_file = est_path / file.name / subfile.name
            if corresponding_est_file.exists() == False:
                print('\t', 'File does not exist in estimated path!')
                continue
            if subfile.suffix in ['.nii', '.gz']:
                compare_images(subfile, corresponding_est_file)
            elif subfile.suffix == '.mat':
                compare_matrices(subfile, corresponding_est_file)
    else:
        print(file)
        corresponding_est_file = est_path / file.name
        if corresponding_est_file.exists() == False:
            print('\t', 'File does not exist in estimated path!')
            continue
        if file.suffix in ['.nii', '.gz']:
            compare_images(file, corresponding_est_file)
        elif file.suffix == '.mat':
            compare_matrices(file, corresponding_est_file)


In [ ]:
# test fslsplit in the three orientations
from fsl.wrappers.avwutils  import fslsplit

inp_path = Path('/Users/Vasilis/Downloads/PSOCT_test_output_3')
out_path = Path('/Users/Vasilis/Downloads/fslsplit_test')
slide_deck = inp_path / 'slide_deck.nii.gz'
orientation = 'sagittal'  

# Lookup table for orientation information
OrientationLookup = {'sagittal': 'x', 'coronal': 'y', 'axial': 'z',
                     'Sagittal': 'x', 'Coronal': 'y', 'Axial': 'z'}

indiv_slides = fslsplit(src=slide_deck, out=out_path/orientation, dim=OrientationLookup[orientation])
